# AWS S3 Vectors: Load Document and Query Embeddings

This notebook embeds policy text using a sentence-transformer model, stores vectors in S3 Vectors, and runs semantic search queries.

<style>
.jp-Notebook .jp-MarkdownCell .jp-RenderedHTMLCommon {
  font-family: "Charter", "Palatino Linotype", "Book Antiqua", Georgia, serif;
  font-size: 17px;
  line-height: 1.7;
}
.jp-Notebook .jp-MarkdownCell .jp-RenderedHTMLCommon h1,
.jp-Notebook .jp-MarkdownCell .jp-RenderedHTMLCommon h2,
.jp-Notebook .jp-MarkdownCell .jp-RenderedHTMLCommon h3 {
  letter-spacing: 0.2px;
  line-height: 1.35;
}
.jp-Notebook .jp-MarkdownCell {
  background: #f8fafc;
  border-left: 4px solid #2563eb;
  border-radius: 6px;
  padding: 4px 10px;
}
</style>

This notebook uses a learning-focused markdown style so explanation cells are easier to scan than code cells.

Import only what we need: AWS client setup, local file reading, and sentence-transformer embedding.

In [ ]:
from pathlib import Path

import boto3
from sentence_transformers import SentenceTransformer

Set AWS and vector index configuration. `MODEL_NAME` controls embedding behavior and output dimension.

In [ ]:
AWS_ACCESS_KEY_ID = "REPLACE_ME"
AWS_SECRET_ACCESS_KEY = "REPLACE_ME"
AWS_REGION = "us-east-1"

BUCKET_NAME = "airline-policy-vectors-YOURNAME"
VECTOR_INDEX_NAME = "airline-policy-index"
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K = 3

Create the AWS S3 Vectors client and load the embedding model once. This keeps later cells focused on data flow.

In [ ]:
session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_REGION,
)
client = session.client("s3vectors")
model = SentenceTransformer(MODEL_NAME)

print("Client and model are ready")

Read the local document and split it into paragraph chunks.

Analogy: this is like cutting a long lecture into short flash cards so each card can be indexed and retrieved independently.

In [ ]:
policy_path = Path("airline_security_policy.txt")
policy_text = policy_path.read_text(encoding="utf-8")
paragraphs = [p.strip() for p in policy_text.split("\n\n") if p.strip()]

print("Paragraphs loaded:", len(paragraphs))
paragraphs[:2]

Generate embeddings for each paragraph and inspect the resulting dimension. The index dimension must match this value exactly.

In [ ]:
embeddings = model.encode(paragraphs)
embedding_vectors = embeddings.tolist()
embedding_dim = len(embedding_vectors[0])

print("Embedding dimension:", embedding_dim)
print("Vectors prepared:", len(embedding_vectors))

Fetch index metadata and detect index dimension safely. Some API responses return fields at different levels, so we read both top-level and nested shapes.

In [ ]:
index_info = client.get_index(
    vectorBucketName=BUCKET_NAME,
    indexName=VECTOR_INDEX_NAME,
)
index_cfg = index_info.get("index") or {}
index_dim = index_info.get("dimension") or index_cfg.get("dimension")
index_dim = index_dim or index_info.get("vectorDimension")
index_dim = index_dim or index_cfg.get("vectorDimension")

if not index_dim:
    raise KeyError(f"Missing dimension in get_index response: {list(index_info.keys())}")
index_dim = int(index_dim)
if index_dim != embedding_dim:
    raise ValueError(
        f"Index dimension {index_dim} != embedding dimension {embedding_dim}. "
        f"Delete index '{VECTOR_INDEX_NAME}' and recreate it with dimension {embedding_dim}."
    )
print("Dimension check passed:", index_dim)

Use the original sentence-transformer vectors for upload.

Analogy: we keep the original photo quality instead of resizing it, because the album frame now matches the photo size.

In [ ]:
storage_vectors = embedding_vectors
print("Vector dimension used for upload:", len(storage_vectors[0]))

Build the upload payload from text and aligned vectors. Each payload item has `key`, `data.float32`, and `metadata.text`.

In [ ]:
vectors_payload = []
for i, text in enumerate(paragraphs, start=1):
    vectors_payload.append(
        {
            "key": f"section-{i}",
            "data": {"float32": storage_vectors[i - 1]},
            "metadata": {"text": text},
        }
    )

print("Payload size:", len(vectors_payload))

Upload vectors to S3 Vectors. On success, you can move straight to semantic query.

In [ ]:
put_response = client.put_vectors(
    vectorBucketName=BUCKET_NAME,
    indexName=VECTOR_INDEX_NAME,
    vectors=vectors_payload,
)

print("put_vectors completed")
put_response

Create a natural-language query, embed it with the same model, and ask S3 Vectors for the nearest matches.

In [ ]:
query_text = "What security checks happen before boarding?"
query_vector = model.encode([query_text])[0].tolist()

query_response = client.query_vectors(
    vectorBucketName=BUCKET_NAME,
    indexName=VECTOR_INDEX_NAME,
    topK=TOP_K,
    queryVector={"float32": query_vector},
    returnMetadata=True,
    returnDistance=True,
)

Show ranked results with short text previews.

Analogy: this works like a recommendation list where the smallest distance means the closest topical match.

In [ ]:
hits = query_response.get("vectors") or []
for rank, row in enumerate(hits, start=1):
    text = (row.get("metadata") or {}).get("text", "")
    preview = text[:140].replace("\n", " ")
    print(f"#{rank} key={row.get('key')} distance={row.get('distance')}")
    print(preview)
    print()